# V3 Category Handler Test Runner

Exercise the new Step 3 → category-handler endpoint path for representative categories. Each endpoint cell builds a sample `CategoryCall` plus its parent Step-2 `Trait`, calls `search_v2.endpoint_fetching.category_handlers.run_handler`, and prints the fired endpoint wrappers, handler buckets, top scored candidates, and elapsed time.

The focused cells below cover the endpoint families currently routed through the category-handler system: entity, franchise, keyword, metadata, award, studio, media type, trending, and semantic.

## Setup

Run this cell first. It imports shared category-handler inputs, opens Postgres / Redis / Qdrant, and makes the common `run_handler` helpers available to every endpoint probe.

In [1]:
import sys, time, asyncio
from datetime import date
from pathlib import Path

# Ensure project root is on the path so imports resolve regardless of where the
# notebook is launched from.
project_root = str(Path.cwd())
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from dotenv import load_dotenv
load_dotenv(Path(project_root) / ".env")

from schemas.enums import Polarity, Role
from schemas.step_2 import Trait
from schemas.step_3 import CategoryCall
from schemas.trait_category import CategoryName
from search_v2.endpoint_fetching.category_handlers.handler import run_handler

# Database connections — mirrors the FastAPI lifespan in api/main.py.
from db.postgres import pool as postgres_pool
from db.redis import init_redis, check_redis
from db.qdrant import check_qdrant, qdrant_client
import db.redis as _redis_module

# Idempotent: re-running the setup cell must not double-open the Postgres pool
# or leak a prior Redis client.
if postgres_pool._closed:
    await postgres_pool.open()
    print("Postgres: pool opened")
else:
    print("Postgres: pool already open")

if _redis_module._redis_client is None:
    await init_redis()
    print(f"Redis:    initialized ({await check_redis()})")
else:
    print(f"Redis:    already initialized ({await check_redis()})")

print(f"Qdrant:   {await check_qdrant()}")

/Users/michaelkeohane/Documents/movie-finder-rag/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ImportError: cannot import name 'run_handler' from 'search_v2.endpoint_fetching.category_handlers.handler' (/Users/michaelkeohane/Documents/movie-finder-rag/search_v2/endpoint_fetching/category_handlers/handler.py)

## Configuration

Shared runtime notes. `run_handler` currently configures the handler LLM internally (`gpt-5.4-mini`, no reasoning), so endpoint probe cells do not pass provider/model parameters.

In [ ]:
today = date.today()

print("Handler LLM: OpenAI gpt-5.4-mini")
print("Reasoning:   none")
print("Verbosity:   low")
print(f"Today:      {today.isoformat()}")

Handler LLM: OpenAI gpt-5.4-mini
Reasoning:   none
Verbosity:   low
Today:      2026-05-04


## Result Formatting Helpers

`print_top_results` bulk-fetches movie cards for scored candidates. `print_handler_result` shows the category-handler output buckets plus any fired endpoint wrappers from the new system.


In [ ]:
from datetime import datetime, timezone
from typing import Iterable, Protocol

from db.postgres import fetch_movie_cards
from schemas.entity_translation import (
    CharacterQuerySpec,
    CharacterTarget,
    PersonQuerySpec,
    PersonTarget,
    TitlePatternQuerySpec,
    TitlePatternTarget,
)

TOP_N = 15


class _Scored(Protocol):
    movie_id: int
    score: float


async def print_top_results(
    candidates: Iterable[_Scored],
    *,
    limit: int = TOP_N,
    sort_desc: bool = True,
) -> None:
    """Print scored movies as ``score  title (year)``, one per line.

    Bulk-fetches card metadata in a single query (per the
    "never query Postgres per-candidate" invariant in AGENTS.md).
    """
    items = list(candidates)
    if not items:
        print("(no scored candidates)")
        return

    if sort_desc:
        items = sorted(items, key=lambda c: c.score, reverse=True)
    items = items[:limit]

    cards = await fetch_movie_cards([c.movie_id for c in items])
    card_by_id = {c["movie_id"]: c for c in cards}

    for cand in items:
        card = card_by_id.get(cand.movie_id)
        if card is None:
            title, year = "<missing card>", "?"
        else:
            title = card["title"] or "<untitled>"
            ts = card["release_ts"]
            year = (
                datetime.fromtimestamp(ts, tz=timezone.utc).year
                if ts is not None else "?"
            )
        print(f"  {cand.score:.3f}  {title} ({year})")


def make_trait(
    *,
    surface_text: str,
    evaluative_intent: str,
    contextualized_phrase: str | None = None,
    role: Role = Role.CARVER,
    polarity: Polarity = Polarity.POSITIVE,
    salience: str = "central",
) -> Trait:
    """Small notebook factory for parent Step-2 Trait context.

    The handler stamps role/polarity from this object onto every fired endpoint
    wrapper. The remaining fields are included because they are required by the
    Step-2 schema, even though run_handler only reads role and polarity today.
    """
    return Trait(
        surface_text=surface_text,
        evaluative_intent=evaluative_intent,
        qualifier_relation="n/a",
        anchor_reference="n/a",
        role_evidence="The sample trait is framed as an eligibility criterion for this notebook probe.",
        role=role,
        polarity=polarity,
        relevance_to_query="This notebook sample treats the trait as the central endpoint behavior under inspection.",
        salience=salience,
        contextualized_phrase=contextualized_phrase or surface_text,
    )


def make_category_call(
    *,
    category: CategoryName,
    expressions: list[str],
    retrieval_intent: str,
) -> CategoryCall:
    return CategoryCall(
        category=category,
        expressions=expressions,
        retrieval_intent=retrieval_intent,
    )


def _print_score_map(title: str, score_map: dict[int, float]) -> None:
    print(f"{title}: {len(score_map)}")
    if score_map:
        top = sorted(score_map.items(), key=lambda item: item[1], reverse=True)[:10]
        for movie_id, score in top:
            print(f"  {score:.3f}  {movie_id}")


def _print_block(label: str, value: str, indent: str = "  ") -> None:
    """Print a multi-line string with the first line on the label line and
    subsequent lines indented under it. Preserves the model's template
    layout instead of escaping newlines into a JSON string."""
    lines = (value or "").splitlines() or [""]
    print(f"{indent}{label}: {lines[0]}")
    for line in lines[1:]:
        print(f"{indent}{' ' * (len(label) + 2)}{line}")


def _print_person_target(t: PersonTarget, idx: int) -> None:
    print(f"  Target #{idx + 1} (person)")
    _print_block("person_exploration", t.person_exploration, indent="    ")
    print(f"    forms: {t.forms}")
    print(f"    person_category: {t.person_category.value}")
    _print_block("prominence_exploration", t.prominence_exploration, indent="    ")
    print(f"    prominence_mode: {t.prominence_mode.value}")


def _print_character_target(t: CharacterTarget, idx: int) -> None:
    print(f"  Target #{idx + 1} (character)")
    _print_block("character_exploration", t.character_exploration, indent="    ")
    print(f"    forms: {t.forms}")
    _print_block("prominence_exploration", t.prominence_exploration, indent="    ")
    print(f"    prominence_mode: {t.prominence_mode.value}")


def _print_title_pattern_target(t: TitlePatternTarget, idx: int) -> None:
    print(f"  Target #{idx + 1} (title-pattern)")
    _print_block("pattern_exploration", t.pattern_exploration, indent="    ")
    print(f"    pattern: {t.pattern!r}")
    print(f"    match_type: {t.match_type.value}")


_TARGET_PRINTERS = {
    PersonTarget: _print_person_target,
    CharacterTarget: _print_character_target,
    TitlePatternTarget: _print_title_pattern_target,
}


def _print_entity_spec(wrapper) -> None:
    """Pretty-print PersonQuerySpec / CharacterQuerySpec / TitlePatternQuerySpec.

    Multi-line exploration template strings are rendered with their
    layout preserved (Films:, Credit per film:, Distinct forms:, ...)
    rather than escaped into JSON. Other fields print on a single line.
    """
    _print_block("query_exploration", wrapper.query_exploration)
    print()
    for idx, target in enumerate(wrapper.targets):
        printer = _TARGET_PRINTERS[type(target)]
        printer(target, idx)
        if idx < len(wrapper.targets) - 1:
            print()


_ENTITY_SPEC_TYPES = (PersonQuerySpec, CharacterQuerySpec, TitlePatternQuerySpec)


async def print_handler_result(handler_result, *, elapsed: float) -> None:
    print(f"Handler completed in {elapsed:.2f}s")
    print(f"Category: {handler_result.category.name if handler_result.category else '(none)'}")
    print()

    print("=" * 60)
    print("FIRED ENDPOINT WRAPPERS")
    print("=" * 60)
    if not handler_result.fired_endpoints:
        print("(none)")
    for route, wrapper in handler_result.fired_endpoints:
        print(f"\n--- {route.value} / {type(wrapper).__name__} ---")
        # Entity specs carry multi-line template strings in their exploration
        # fields; rendering them through model_dump_json escapes the layout
        # into "\n" sequences and defeats the format-locking we rely on.
        if isinstance(wrapper, _ENTITY_SPEC_TYPES):
            _print_entity_spec(wrapper)
        else:
            print(wrapper.model_dump_json(indent=2))

    print("\n" + "=" * 60)
    print("HANDLER BUCKETS")
    print("=" * 60)
    _print_score_map("inclusion_candidates", handler_result.inclusion_candidates)
    _print_score_map("downrank_candidates", handler_result.downrank_candidates)
    print(f"exclusion_ids: {len(handler_result.exclusion_ids)}")
    print(f"preference_specs: {len(handler_result.preference_specs)}")

    print("\n" + "=" * 60)
    print("TOP INCLUSION CANDIDATES")
    print("=" * 60)
    pseudo_scores = [type("Scored", (), {"movie_id": mid, "score": score})() for mid, score in handler_result.inclusion_candidates.items()]
    await print_top_results(pseudo_scores)


## Endpoint 1: Entity via Category Handler

Runs a sample `NAMED_CHARACTER` category call through the new handler system. The handler LLM emits an `EntityEndpointParameters` wrapper whose `.parameters` payload is one of the entity executor specs, then the handler executes it as a candidate-generating positive carver.


In [23]:
entity_trait = make_trait(
    surface_text="The Rock",
    evaluative_intent="Identify and retrieve movies featuring the actor Dua Lipa as a primary cast member.",
)
entity_call = make_category_call(
    category=CategoryName.NAMED_CHARACTER,
    expressions=["Gandalf"],
    retrieval_intent=(
        "Identify movies where the character Gandalf appears. This is a direct character lookup to surface films featuring this character across his various credited titles."
    ),
)

start = time.perf_counter()
entity_handler_result = await run_handler(
    category_call=entity_call,
    trait=entity_trait,
    qdrant_client=qdrant_client,
)
entity_elapsed = time.perf_counter() - start
await print_handler_result(entity_handler_result, elapsed=entity_elapsed)


Handler completed in 2.60s
Category: NAMED_CHARACTER

FIRED ENDPOINT WRAPPERS

--- entity / CharacterQuerySpec ---
  query_exploration: One character under several variants.

  Target #1 (character)
    character_exploration: Films: The Lord of the Rings: The Fellowship of the Ring (2001), The Lord of the Rings: The Two Towers (2002), The Lord of the Rings: The Return of the King (2003), The Hobbit: An Unexpected Journey (2012), The Hobbit: The Desolation of Smaug (2013), The Hobbit: The Battle of the Five Armies (2014)
                           Credit per film:
                             - The Lord of the Rings: The Fellowship of the Ring: Gandalf
                             - The Lord of the Rings: The Two Towers: Gandalf
                             - The Lord of the Rings: The Return of the King: Gandalf
                             - The Hobbit: An Unexpected Journey: Gandalf
                             - The Hobbit: The Desolation of Smaug: Gandalf
                          

## Endpoint 2: Franchise via Category Handler

Runs a sample `FRANCHISE_LINEAGE` category call through the new handler system. The LLM emits a `FranchiseEndpointParameters` wrapper, and the handler executes the franchise endpoint.


In [ ]:
franchise_trait = make_trait(
    surface_text="Shrek movies",
    evaluative_intent="The user wants movies belonging to the Shrek franchise or shared universe, with main-line lineage preferred where applicable.",
)
franchise_call = make_category_call(
    category=CategoryName.FRANCHISE_LINEAGE,
    expressions=["Shrek franchise", "main-line Shrek lineage"],
    retrieval_intent=(
        "Search for movies in the Shrek franchise / shared universe. Treat the "
        "franchise name as the primary lookup target and preserve the main-line "
        "lineage preference as a scoring bias rather than excluding related entries."
    ),
)

start = time.perf_counter()
franchise_handler_result = await run_handler(
    category_call=franchise_call,
    trait=franchise_trait,
    qdrant_client=qdrant_client,
)
franchise_elapsed = time.perf_counter() - start
await print_handler_result(franchise_handler_result, elapsed=franchise_elapsed)


## Endpoint 3: Keyword via Category Handler

Runs a sample `GENRE` category call through the handler system. The handler may emit keyword and/or semantic endpoint wrappers depending on the category bucket decision.

In [3]:
keyword_trait = make_trait(
    surface_text="animated movies for kids",
    evaluative_intent="The user wants animated movies suitable for children, with animation as the genre/format identity under inspection in this probe.",
)
keyword_call = make_category_call(
    category=CategoryName.GENRE,
    expressions=["animated movies"],
    retrieval_intent=(
        "Search for animated movies. Treat animation as the genre/format signal "
        "this endpoint probe is inspecting; child suitability is context for the "
        "example rather than a separate endpoint call in this cell."
    ),
)

start = time.perf_counter()
keyword_handler_result = await run_handler(
    category_call=keyword_call,
    trait=keyword_trait,
    qdrant_client=qdrant_client,
)
keyword_elapsed = time.perf_counter() - start
await print_handler_result(keyword_handler_result, elapsed=keyword_elapsed)

NameError: name 'make_trait' is not defined

## Endpoint 4: Metadata via Category Handler

Runs a sample `NUMERIC_RECEPTION_SCORE` category call through the new handler system. The LLM emits a `MetadataEndpointParameters` wrapper whose payload can populate one or more structured metadata columns.


In [ ]:
metadata_trait = make_trait(
    surface_text="low-rated or poorly reviewed",
    evaluative_intent="The user wants movies with weak numeric reception signals, such as low critic or audience ratings.",
)
metadata_call = make_category_call(
    category=CategoryName.NUMERIC_RECEPTION_SCORE,
    expressions=["low audience rating", "poor critic reception"],
    retrieval_intent=(
        "Search for movies with low or poor numeric reception. The expressions "
        "are substitutable reception signals, so a movie can satisfy the call via "
        "low audience rating, poor critic reception, or both."
    ),
)

start = time.perf_counter()
metadata_handler_result = await run_handler(
    category_call=metadata_call,
    trait=metadata_trait,
    qdrant_client=qdrant_client,
)
metadata_elapsed = time.perf_counter() - start
await print_handler_result(metadata_handler_result, elapsed=metadata_elapsed)


## Endpoint 5: Award via Category Handler

Runs a sample `AWARDS` category call through the new handler system. The LLM emits an `AwardEndpointParameters` wrapper, and the handler executes the award endpoint.


In [ ]:
award_trait = make_trait(
    surface_text="won at least one Oscar",
    evaluative_intent="The user wants movies with at least one Academy Award win.",
)
award_call = make_category_call(
    category=CategoryName.AWARDS,
    expressions=["Oscar winner", "at least one win"],
    retrieval_intent=(
        "Search for movies that won at least one Oscar / Academy Award. This is "
        "one award condition with a winner outcome and a minimum count of one."
    ),
)

start = time.perf_counter()
award_handler_result = await run_handler(
    category_call=award_call,
    trait=award_trait,
    qdrant_client=qdrant_client,
)
award_elapsed = time.perf_counter() - start
await print_handler_result(award_handler_result, elapsed=award_elapsed)


## Endpoint 6: Studio via Category Handler

Runs a sample `STUDIO_BRAND` category call through the new handler system. The LLM emits a `StudioEndpointParameters` wrapper, and the handler executes the studio endpoint.


In [ ]:
studio_trait = make_trait(
    surface_text="Disney films",
    evaluative_intent="The user wants movies produced by the Disney studio brand / production company family.",
)
studio_call = make_category_call(
    category=CategoryName.STUDIO_BRAND,
    expressions=["Disney"],
    retrieval_intent=(
        "Search for films produced by Disney as a production-company / studio "
        "brand. Treat Disney as an umbrella studio-brand lookup, not streaming "
        "availability on Disney+."
    ),
)

start = time.perf_counter()
studio_handler_result = await run_handler(
    category_call=studio_call,
    trait=studio_trait,
    qdrant_client=qdrant_client,
)
studio_elapsed = time.perf_counter() - start
await print_handler_result(studio_handler_result, elapsed=studio_elapsed)


## Endpoint 7: Media Type via Category Handler

Runs a sample `MEDIA_TYPE` category call through `run_handler`. Media type remains deterministic under the hood, but the notebook now exercises it through the same handler entry point as the LLM-backed categories.

In [ ]:
media_type_trait = make_trait(
    surface_text="TV movies and shorts",
    evaluative_intent="The user wants non-default media formats, specifically television movies and short films.",
)
media_type_call = make_category_call(
    category=CategoryName.MEDIA_TYPE,
    expressions=[
        "TV movies",
        "shorts",
    ],
    retrieval_intent=(
        "Identify movies whose primary distribution format is television or short "
        "film. This call gates the population to include only explicit non-default "
        "media formats, distinguishing them from standard theatrical feature films."
    ),
)

start = time.perf_counter()
media_type_handler_result = await run_handler(
    category_call=media_type_call,
    trait=media_type_trait,
    qdrant_client=qdrant_client,
)
media_type_elapsed = time.perf_counter() - start
await print_handler_result(media_type_handler_result, elapsed=media_type_elapsed)

## Endpoint 8: Trending via Category Handler

Runs a sample `TRENDING` category call through `run_handler`. Trending remains deterministic under the hood and reads the precomputed Redis signal, but it now uses the shared handler return contract.

In [ ]:
trending_trait = make_trait(
    surface_text="trending right now",
    evaluative_intent="The user wants movies with live current-buzz / trending status rather than static popularity.",
)
trending_call = make_category_call(
    category=CategoryName.TRENDING,
    expressions=["trending right now"],
    retrieval_intent=(
        "Retrieve movies that are currently trending or broadly being watched "
        "right now. This is a live popularity signal, not static all-time appeal."
    ),
)

start = time.perf_counter()
trending_handler_result = await run_handler(
    category_call=trending_call,
    trait=trending_trait,
    qdrant_client=qdrant_client,
)
trending_elapsed = time.perf_counter() - start
await print_handler_result(trending_handler_result, elapsed=trending_elapsed)

## Endpoint 9a: Semantic Dealbreaker via Category Handler

Runs a positive `CARVER` semantic-style category call through `run_handler`, using `NARRATIVE_DEVICES` as the representative category.

In [2]:
semantic_dealbreaker_trait = make_trait(
    surface_text="unreliable narrator",
    evaluative_intent="The user wants movies where an unreliable narrator is a central storytelling device.",
)
semantic_dealbreaker_call = make_category_call(
    category=CategoryName.ELEMENT_PRESENCE,
    expressions=["clowns"],
    retrieval_intent=(
        "The search should identify movies where clowns are a concrete and identifiable element. This includes films where clowns appear as characters, visual motifs, or central sources of horror, ensuring the presence of the 'clown' element is a primary signal for the result pool."
    ),
)

start = time.perf_counter()
semantic_dealbreaker_handler_result = await run_handler(
    category_call=semantic_dealbreaker_call,
    trait=semantic_dealbreaker_trait,
    qdrant_client=qdrant_client,
)
semantic_dealbreaker_elapsed = time.perf_counter() - start
await print_handler_result(
    semantic_dealbreaker_handler_result,
    elapsed=semantic_dealbreaker_elapsed,
)

NameError: name 'make_trait' is not defined

## Endpoint 9b: Semantic Preference via Category Handler

Runs a positive `QUALIFIER` semantic-style category call through `run_handler`, using `EMOTIONAL_EXPERIENTIAL` as the representative preference category. Positive qualifier outputs are returned as deferred preference specs by the handler.

In [ ]:
semantic_preference_trait = make_trait(
    surface_text="slow-burn, atmospheric, and melancholy for a rainy Sunday afternoon",
    evaluative_intent="The user prefers a slow-burn, atmospheric, melancholy viewing mood appropriate for a quiet rainy Sunday afternoon.",
    role=Role.QUALIFIER,
)
semantic_preference_call = make_category_call(
    category=CategoryName.EMOTIONAL_EXPERIENTIAL,
    expressions=["slow-burn", "atmospheric", "melancholy"],
    retrieval_intent=(
        "Score for a slow-burn, atmospheric, melancholy viewing experience. "
        "Treat these as mood and pacing preferences rather than hard population "
        "requirements."
    ),
)

start = time.perf_counter()
semantic_preference_handler_result = await run_handler(
    category_call=semantic_preference_call,
    trait=semantic_preference_trait,
    qdrant_client=qdrant_client,
)
semantic_preference_elapsed = time.perf_counter() - start
await print_handler_result(
    semantic_preference_handler_result,
    elapsed=semantic_preference_elapsed,
)